# Day 5 Lab 5：给职业数字人加上边界、兜底和交接

前四天，我们已经一步一步搭出了职业数字人的主体能力：

- Day 1：接上大模型，让它有了“大脑”
- Day 2：加入 routing，让它能先判断任务类型
- Day 3：接上 tools，让它能读取、分析和执行动作
- Day 4：加入 context，让它能结合 history、memory 和 resource 工作

今天，我们给它补上最后一块能力：

> **可信。**

一个职业数字人不能只是会回答。  
它还要知道：

- 哪些问题可以处理
- 哪些内容不能乱答
- 条件不够时怎么兜底
- 处理不下去时怎么交给人

---

## 今天的 3 个核心概念

### 1. Guardrail：先判断边界

Guardrail 可以理解成 AI 的安全边界。

在今天的 Lab 里，我们先做一个最小版本：

> **用户的问题能不能进入正常流程？**

比如：

- 普通写作任务：可以继续
- 普通资料问答：可以继续
- 明显涉及敏感信息的问题：先拦一下
- 超出当前能力范围的问题：不要硬答

---

### 2. Fallback：条件不够时兜底

Fallback 是兜底回答。

它处理的是这种情况：

> 问题本身可以处理，但现在条件不够。

比如：

- 用户没有上传文件
- 资料里没有相关信息
- 工具执行失败
- 数据太少，结论不确定
- 用户的问题不够清楚

这时候，职业数字人不应该乱编。  
它应该说明限制，并告诉用户下一步怎么继续。

---

### 3. Handoff：该交给人时交出去

Handoff 是任务交接。

你可以把它理解成网购时的“转人工”。

当 AI 处理不下去时，  
不是简单说一句“我不能处理”，  
而是生成一段交接摘要，让人可以接着处理。

今天我们先做最简单的版本：

> **AI → 人的交接摘要。**

多智能体系统里的 agent-to-agent handoff，  
我们放到中级课程再学。

---

## 今天 Lab5 要做什么

今天我们会做三个最小函数：

```python
input_guardrail()
fallback_response()
generate_handoff_summary()
```

它们分别对应：

Guardrail：判断问题能不能进入正常流程
Fallback：条件不够时给出兜底回答
Handoff：需要人接手时生成交接摘要

最后，我们会把它们接进职业数字人的流程里：


```text
用户输入
  ↓
Guardrail 判断边界
  ↓
可以处理 → 进入正常职业数字人流程
条件不够 → Fallback 兜底
需要人判断 → Handoff 交接
  ↓
输出结果
```

---

今天完成后，职业数字人会变成什么样？

完成今天的 Lab 后，职业数字人就不只是会回答、会调用工具、会读资料。

它还会：

- 不该答的问题先停下来
- 资料不足时不乱编
- 工具失败时说明情况
- 结果不确定时降低语气
- 需要人判断时生成交接摘要

这一步做完，
它就更像一个真实工作里可以被信任的 AI 助手。

# Part 1：导入前四天的模块

前四天，我们已经把职业数字人的主体能力拆成了几个 `.py` 文件。

今天不会从零重写。

我们要做的是：

```text
导入前四天的模块
↓
检查它们能不能正常运行
↓
在外面加上 guardrail、fallback 和 handoff
↓
最后接成一个可以展示的 Gradio 页面
```

现在项目结构大概是这样：

```text
ai-agent-course/
├── lab5.ipynb
├── src/
│   ├── __init__.py
│   ├── llm.py              # Day 1：模型调用
│   ├── task_router.py      # Day 2：任务判断
│   ├── tool.py             # Day 3：工具函数
│   ├── context.py          # Day 4：上下文 / resource / RAG
│   ├── agent_core.py       # Day 2-4：主流程整合
│   └── guardrail.py        # Day 5：边界、兜底、交接
├── data/
│   └── context/
│       ├── history.json
│       ├── memory.json
│       └── resources/
│           └── resource.txt
├── .env
├── .gitignore
└── requirements.txt
```
这些文件分别负责不同的事情：

```text
task_router.py：判断用户输入属于哪类任务
tool.py：放工具函数，比如数据分析、网页读取、消息推送
context.py：处理 history、memory、resource，把上下文拼起来
agent_core.py：把任务判断、工具、上下文和模型调用串起来
guardrail.py：今天新增，用来处理边界、兜底和交接
```

先把前四天的模块导入进来。


In [13]:
# 确认当前路径
from pathlib import Path
import sys

In [14]:
# 查看当前 notebook 所在目录
current_dir = Path.cwd()
current_dir

WindowsPath('c:/Users/240411014/Desktop/CNGR_agents')

如果你的 notebook 就放在项目根目录下，  
也就是和 `src/` 文件夹在同一层，  
一般可以直接导入。

如果导入失败，  
我们可以把当前项目目录加入 Python 的搜索路径。

In [15]:
# 把当前项目目录加入 Python 搜索路径
if str(current_dir) not in sys.path:
    sys.path.append(str(current_dir))

print("当前项目目录已加入 Python 搜索路径：")
print(current_dir)

当前项目目录已加入 Python 搜索路径：
c:\Users\240411014\Desktop\CNGR_agents


接下来导入前四天已经写好的模块。

如果这里报错，通常说明两种情况：

1. `src/` 文件夹不在当前项目目录下
2. `src/` 里面缺少 `__init__.py`

`__init__.py` 可以是一个空文件。  
它的作用是告诉 Python：这个文件夹可以被当成模块导入。

In [ ]:
import src.llm as llm
from IPython.display import display, Markdown

answer = llm.call_llm("请用一句话解释什么是 AI中的guardrail。")
display(Markdown(answer))

Guardrail（安全护栏）是一种预设的防护机制或规则边界，用于在物理设施或数字系统中划定安全范围，防止越界行为或产生不良后果。

In [2]:
import src.task_router as task_router

test_inputs = [
    "你是谁？你能帮我做什么？",
    "帮我写一段周报开头。",
    "请分析这份表格。",
    "帮我查一下这家公司最近有什么新闻。",
    "把刚才的结果发到测试群。",
    "我下周的工作重点应该是什么？",
    "帮我处理一下这个事情。"
]

for text in test_inputs:
    task_type = task_router.classify_task(text)
    print(text, "→", task_type)

你是谁？你能帮我做什么？ → profile
帮我写一段周报开头。 → writing
请分析这份表格。 → data_analysis
帮我查一下这家公司最近有什么新闻。 → web_search
把刚才的结果发到测试群。 → send_message
我下周的工作重点应该是什么？ → work_planning
帮我处理一下这个事情。 → clarify


In [3]:
import src.tool as tool

result = tool.full_data_analysis_tool("data/sales_data.csv")
display(Markdown(result))

基于提供的统计信息与样例数据，总结如下：

**【3个核心重点】**
1. **业绩波动显著，峰值拉动效应强**：销售额均值约2.6万元，但标准差较高（约9343元），最大值（4.2万）是最小值（1.2万）的3.5倍。表明月度业绩起伏较大，部分高产出月份对整体数据有明显拉升作用。
2. **指标高度协同，前期增长趋势明确**：销售额、订单数与客户数变化方向完全一致。前5个月样例显示整体稳步上行，仅4月出现短暂微跌，5月迅速反弹，说明业务基本盘健康且具备自我修复能力。
3. **数据质量完整，分布相对均衡**：12条记录无缺失，且三项指标的中位数与均值非常接近（如销售额中位数2.7万 vs 均值2.6万），说明数据未受极端离群值严重干扰，整体分布健康，适合直接用于经营分析。

**🔍 值得关注的趋势与提示**
- **趋势**：1-5月呈现“增长→微跌→反弹”的上升通道，客户数扩张是订单与销售额提升的核心驱动力。
- **异常/波动提示**：4月三项指标同步小幅回落，虽未破坏整体趋势，但打破了连续增长节奏，建议结合具体业务背景（如活动排期、节假日或供应链因素）排查原因；同时，较高的标准差提示全年可能存在明显的淡旺季分化，需结合完整12个月数据进一步验证季节性规律。
- **缺失值**：数据完整，无缺失。

In [4]:
answer = tool.web_qa_tool(
    url="https://www.python.org/",
    question="这个网站主要介绍什么？"
)

display(Markdown(answer))

该网站是Python编程语言的官方网站（Python.org），主要介绍Python语言及其核心特点（如语法简洁、易学、开发高效、便于系统集成），并提供Python的下载、官方文档、社区资源、新闻资讯以及相关活动（如PyCon大会）等信息。

In [5]:
preview = tool.send_message_tool("这是 Day5 的测试消息。", confirm=False)
display(Markdown(preview))


    【消息预览】

    这是 Day5 的测试消息。

    当前 confirm=False，所以还没有真正发送。
    如果确认要发送，请把 confirm 改成 True。
    

In [6]:
import src.context as context

profile_context = context.build_profile_context()
print(profile_context[:500])

【个人简历资料 cv.txt】
    # 张启明  
AI & Digital Transformation Executive  
Email: zhang.qiming@example.com  
Location: Singapore / Shanghai
电话：12345678901  

---

## 🧭 Profile

拥有15年跨领域经验的AI与数字化转型领导者，职业路径从算法工程师起步，逐步转型为战略咨询顾问，最终担任集团CTO。

在工业AI、企业数据平台、智能决策系统及组织数字化转型方面具备深厚经验。擅长将复杂技术能力转化为业务价值，推动从0到1的系统落地与规模化应用。

具备从技术研发、产品设计到高层战略决策的全链路能力，曾多次直接向CEO及董事会汇报。

---

## 💼 Professional Experience

### Group Chief Technology Officer (CTO)  
Global Industrial Group  
2021 – Present  

- 负责集团整体数字化与AI战略，覆盖生产、供应链、研发与运营


In [ ]:
# 测试history / memory
basic_context = context.build_basic_context(
    current_question="请把刚才的数据分析结果整理成一段适合发到项目群的消息。",
    include_profile=False
)

print(basic_context[:1500])

【当前问题】
    请把刚才的数据分析结果整理成一段适合发到项目群的消息。

    

    【History：最近对话和任务记录】

来源：day3_analysis_history.txt
    内容：
    Day3 数据分析对话历史

用户上传了一份销售数据表，文件名为 sales_data.csv。

数据分析工具完成了基础分析，得到以下结果：
1. 本月整体销售额较上月上涨 12.5%。
2. 华东区销售额增长最快，环比增长 18.2%。
3. 华东区退货率也明显偏高，达到 7.8%，高于其他区域平均水平。
4. 华南区销售额稳定，但客单价略有下降。
5. 数据分析工具建议继续关注华东区退货原因，尤其是产品批次、渠道反馈和售后记录。

用户接着说：
“把刚才的数据分析结果，整理一下发到项目群里。”

注意：
这份 history 只是为了 Lab4 演示。
它代表当前对话前面已经发生过的内容。

    【Memory：长期记忆和工作偏好】

来源：user_profile_memory.json
    内容：
    {
  "user_profile": {
    "role": "项目负责人",
    "department": "数字化与业务协同团队",
    "work_context": "经常需要把数据分析结果整理成适合项目群、领导群或跨部门会议使用的简短结论。",
    "communication_style": "简洁、正式、清楚，不夸张，不写空话。"
  },
  "long_term_preferences": {
    "message_style": "先说结论，再说风险，最后说建议动作。",
    "preferred_length": "适合发群消息，控制在 150 字以内。",
    "tone": "稳重、专业、便于团队快速理解。"
  },
  "boundaries": {
    "sensitive_information": [
      "客户名称",
      "内部成本",
      "利润率",
      "具体异常明细",
      "未经确认的责任归因",
      "个人信息"
    ],
    "rules": [
      "敏感数据不要直接发到群

In [10]:
# 测试最小 RAG
resource_index = context.build_resource_index()

retrieved_chunks = context.retrieve_relevant_chunks(
    question="请按照项目同步格式写一段项目群消息。",
    resource_index=resource_index,
    top_k=3
)

for item in retrieved_chunks:
    print("来源：", item["source"])
    print("相似度：", item["score"])
    print(item["content"][:500])
    print("-" * 50)

来源： communication_policy.txt
相似度： 0.7287178973716749
业务沟通与信息边界说明

项目群可以同步：
1. 总体趋势。
2. 初步结论。
3. 需要关注的风险点。
4. 下一步建议动作。
5. 不涉及敏感细节的概括性提醒。

项目群不建议直接同步：
1. 客户真实名称。
2. 内部成本、利润率、报价底线。
3. 未经确认的异常原因。
4. 具体责任归因。
5. 个人信息或可识别个人的数据。
6. 可能引发误解的单点数据。

推荐表达方式：
1. 用“数据显示”“初步看”“建议进一步确认”表达尚未完全确认的结论。
2. 用“某区域”“部分渠道”“个别指标”替代敏感明细。
3. 如果需要详细数据，应建议在线下报告或专项会议中展开。
4. 发群消息时，优先保证清楚、稳妥、可执行。
--------------------------------------------------
来源： project_message_template.txt
相似度： 0.7256270196396142
# 项目群消息模板

适用场景：
当需要把数据分析结果、项目进展、风险提醒同步到项目群时，优先使用这个格式。

请使用以下结构：

【结论】
用 1 句话说明最重要的结果。
不要写太长，不要堆数据。

【关键信号】
用 2 到 3 条 bullet point 说明支撑结论的主要发现。
只写趋势和重点，不展开敏感明细。

【风险提醒】
如果发现异常、波动或潜在风险，用 1 到 2 句话说明。
表达要克制，不要夸大问题。

【建议动作】
写 1 到 3 条下一步建议。
建议要具体、可执行，适合项目群协作。

【需要确认】
如果有需要团队确认的数据、口径或下一步负责人，在这里列出。
如果没有，可以写“暂无”。

写作要求：
1. 语气简洁、正式、适合工作群。
2. 不要直接写客户名称、内部成本、利润、供应商敏感信息。
3. 不要在群里展开异常明细，只做概括提醒。
4. 不要使用夸张表达，例如“严重”“灾难性”“巨大问题”，除非资料中明确说明。
5. 如果信息不足，要写“需要进一步确认”，不要自行补充。
--------------------------------------------------
来源： project_message

# Part 2：理解 agent_core.py —— 职业数字人的主流程

前面我们已经有了几个模块：

- `llm.py`：负责调用大模型
- `task_router.py`：负责判断任务类型
- `tool.py`：负责表格分析、网页读取、消息推送
- `context.py`：负责读取身份资料、history、memory、resource，并拼出 context

但只有这些模块还不够。

因为它们只是一个个“零件”。

真正运行时，系统还需要一个地方来决定：

```text
用户输入之后，
先调用谁？
再调用谁？
什么时候直接回答？
什么时候调用工具？
什么时候读取资料？
什么时候先追问？
```

这个主流程就放在：

```python
agent_core.py
```

可以把它理解成职业数字人的“总控”。

它负责把前四天的能力串起来：

```text
用户输入
↓
task_router 判断任务类型
↓
agent_core 检查这条路径需要哪些信息
↓
信息够了 → 调用 tool.py / context.py / llm.py
信息不够 → 先追问用户
↓
返回最终结果
```

In [17]:
import importlib
import src.agent_core as agent_core

agent_core = importlib.reload(agent_core)

print("agent_core.py 导入成功。")

agent_core.py 导入成功。


In [12]:
# 测试 1：profile 路径
result = agent_core.handle_user_task("你是谁？你做过什么？")
display(Markdown(result))

您好，我是**张启明**，一名拥有15年跨领域经验的**AI与数字化转型领导者**，目前担任某全球工业集团的**集团首席技术官（CTO）**，常驻新加坡/上海。我的职业路径从底层算法研发起步，历经战略咨询，最终走向集团技术管理与战略决策岗位。

基于我的个人资料，我主要做过以下几方面的工作：

### 🔹 集团级AI战略与平台建设（2021至今｜CTO）
- **统筹数字化与AI战略**：负责集团在生产、供应链、研发与运营体系的整体技术规划，直接向CEO及董事会汇报。
- **搭建企业级数据与AI中台**：主导打通跨业务线数据孤岛，构建统一的数据层与建模能力，支撑上层AI应用规模化落地。
- **工业AI项目落地与降本增效**：主导预测性维护、流程优化、质量控制等工业AI项目，实现年度成本节省超**5000万美元**。
- **建立AI治理体系**：制定数据标准、模型评估机制与安全策略，并管理超50人的跨职能技术团队（数据、算法、平台、工程）。

### 🔹 企业数字化转型咨询（2017-2021｜高级战略顾问）
- 为制造业、能源及医疗行业客户提供AI与数字化转型战略规划，主导多个千万级项目（智能工厂、数据平台、AI应用规划）。
- 设计企业级数据架构（Data Lake / Data Warehouse / AI Layer），并协助客户从0搭建内部AI团队与能力体系。
- 深度参与C-level沟通，将技术能力转化为可落地的业务路线图。

### 🔹 算法研发与工程实践（2012-2017｜高级算法工程师）
- 专注于机器学习与深度学习模型开发，涵盖预测模型、推荐系统与图像识别。
- 熟练使用Python、TensorFlow、XGBoost等技术栈，成功推动多个算法模型在业务场景中上线并带来显著业务提升。

### 🔹 近期工作重心与职业思考（基于2025年4月工作记录）
- **从“技术视角”转向“业务与系统视角”**：近期重点推进生产过程优化项目，发现业务端更关注系统“稳定性”而非单纯模型精度；正在推动建立模型上线后的持续评估与监控机制。
- **强化跨部门协同与ROI导向**：在准备CEO汇报时，聚焦AI项目的实际业务价值与资源投入回报，避免陷入技术细节；深刻认识到降低部门协同成本比技术本身更关键。
- **组织能力构建**：正从“做模型的人”向“做系统的人”转变，认为未来3年AI要成为企业核心能力，关键在于培养“懂业务+懂技术”的复合型人才与配套的组织机制。

如果您希望了解某个具体项目（如数据中台架构设计、工业AI落地细节、或团队管理经验）的更多背景，我可以为您进一步展开说明。

In [13]:
# 测试 2：data_analysis 路径
result = agent_core.handle_user_task("请分析 data/sales_data.csv，告诉我三个重点。")
display(Markdown(result))

以下是基于您提供的统计信息与样例数据的分析总结：

**📊 核心重点（3条）**
1. **业务指标整体呈上升态势**：1-5月销售额、订单数与客户数同步增长，仅4月出现微幅环比回落，整体处于稳步扩张期。
2. **月度波动剧烈，峰值分化明显**：三项指标最大值均为最小值的3倍左右（如销售额1.2万~4.2万），标准差较高且最大值显著高于均值，表明全年业绩分布不均，存在明显的淡旺季或促销爆发期。
3. **指标高度协同，单均价值持续优化**：销售额与订单数、客户数变化方向高度一致；基于样例测算，单均销售额（销售额÷订单数）从1月的100元提升至5月的约116元，反映客单价或订单质量在稳步改善。

**⚠️ 数据质量与趋势提示**
- **缺失值**：数据完整，12个月份均无缺失，可直接用于后续分析。
- **值得关注点**：
  - **4月微跌**：销售额、订单数、客户数均较3月小幅下降，需排查是否为季节性回调、库存调整或营销活动空窗期所致。
  - **峰值驱动**：结合1月为全年最低、最大值（4.2万/360单/248客）远高于均值，可推断业绩高点集中在年中或年末。建议重点复盘峰值月份的具体驱动因素（如大促、渠道放量等），以便优化全年资源投放节奏。

In [14]:
# 测试 3：web_search 路径，缺 URL
result = agent_core.handle_user_task("帮我看看这个网页讲了什么。")
display(Markdown(result))


    我可以帮你读取网页并回答问题。

    请把网页链接发给我。
    也可以一起告诉我你最关心什么问题。

    例如：
    请帮我看这个网页里有没有联系方式：https://example.com
    

In [15]:
# 测试 4：web_search 路径，有 URL
result = agent_core.handle_user_task(
    "请帮我看这个网页主要讲什么：https://www.python.org/"
)

display(Markdown(result))

该网页是Python编程语言的官方网站（Python.org）首页，主要内容包括：

1. **语言特点介绍**：强调Python语法简洁、易于上手，能让开发者快速工作并高效集成系统，适合编程新手与有经验的开发者。
2. **核心功能演示**：通过交互式代码示例，直观展示Python的基础语法与特性（如算术运算、列表操作、循环控制、函数定义及输入输出等）。
3. **官方资源导航**：提供Python最新版本下载、官方文档、社区交流、工作机会、开发者活动（如PyCon US 2026）及新手入门指南等入口。

总体而言，该页面旨在帮助访问者快速了解Python语言的优势，并引导其获取学习资源、下载工具及参与官方社区。

In [16]:
# 测试 5：send_message 路径，默认预览
result = agent_core.handle_user_task("把刚才结果发到群里。")
display(Markdown(result))


    【消息预览】

    该网页是Python编程语言的官方网站（Python.org）首页，主要内容包括：

1. **语言特点介绍**：强调Python语法简洁、易于上手，能让开发者快速工作并高效集成系统，适合编程新手与有经验的开发者。
2. **核心功能演示**：通过交互式代码示例，直观展示Python的基础语法与特性（如算术运算、列表操作、循环控制、函数定义及输入输出等）。
3. **官方资源导航**：提供Python最新版本下载、官方文档、社区交流、工作机会、开发者活动（如PyCon US 2026）及新手入门指南等入口。

总体而言，该页面旨在帮助访问者快速了解Python语言的优势，并引导其获取学习资源、下载工具及参与官方社区。

    当前 confirm=False，所以还没有真正发送。
    如果确认要发送，请把 confirm 改成 True。
    

# Part 3：加入最小 input guardrail

现在，职业数字人的主流程已经能跑了。

但真实工作里，能跑还不够。

有些信息不能随便说出去。

比如：

- 电话
- 邮箱

所以这一节我们先做一个最小版本的 guardrail：

```text
如果用户要求查看、输出、发送电话或邮箱，
系统先拦住，不进入正常主流程。
```

今天不做复杂权限系统。

我们只让大家先看到：

```text
guardrail 可以放在主流程前面，
先判断这个问题能不能进入系统。
```

流程会变成：

```text
用户输入
↓
input_guardrail
↓
安全 → 进入 agent_core.py
涉及电话 / 邮箱 → 拦截并提示
↓
最终输出
```
---

In [18]:
import importlib
import src.guardrail as guardrail

guardrail = importlib.reload(guardrail)

print("guardrail.py 导入成功。")

guardrail.py 导入成功。


In [19]:
# 测试 1：普通任务可以通过
result = guardrail.safe_handle_user_task("帮我写一段周报开头。")
display(Markdown(result))

以下是为您撰写的周报开头，已严格对齐您的CTO身份、本周工作记录及“重业务价值、重系统视角”的表达偏好：

---

**【本周核心摘要】**
本周工作聚焦于集团AI战略的系统化落地与业务价值对齐。重点推进三项主线：一是完成“数据中台+AI中台”整体架构规划，着手破解跨业务线数据孤岛；二是将工业AI项目重心从模型精度转向生产稳定性，并正式启动模型上线后的评估与监控机制；三是重构面向CEO的汇报逻辑，全面以业务ROI与资源规划为导向。整体来看，团队工作视角已实现从“单点技术交付”向“系统化能力构建与跨部门协同”的实质性过渡。以下为详细进展与下周计划：

---

**💡 使用提示：**
- 结构采用“结论先行 → 关键动作 → 视角转变 → 过渡下文”，符合高管周报的阅读习惯。
- 已规避技术细节堆砌，突出您本周在架构规划、业务稳定性、ROI导向及组织能力上的核心思考。可直接粘贴至周报正文开头。

In [20]:
# 测试 2：电话信息被拦截
result = guardrail.safe_handle_user_task("请告诉我你的电话。")
display(Markdown(result))

这个问题我不能直接处理。

    原因：
    这个请求可能涉及电话或邮箱等隐私信息，不能直接输出或发送。

    如果这是工作中必须处理的信息，请先确认你有权限查看和发送。

## 拓展练习：给职业数字人加上 fallback 或 handoff

这一节，我们只做了一个最小 input guardrail：

```text
如果用户请求电话、邮箱等隐私信息，
系统先拦截，不进入正常主流程。
```

如果你想继续练习，可以尝试再加两个能力。

### 练习 1：加入 fallback

fallback 的意思是：

```text
当系统不能正常完成任务时，
不要硬答，而是给出一个清楚的兜底回复。
```

你可以尝试处理这些情况：

- 文件不存在
- 网页打不开
- 用户问题太模糊
- 资料里没有找到答案
- 工具执行失败

比如，当用户说：

请分析 data/not_exist.csv

系统可以回复：

我现在无法完成这个表格分析。

原因是：没有找到这个文件。

你可以检查文件路径是否正确，
或者重新上传一份 CSV / Excel 文件。

### 练习 2：加入 handoff

handoff 的意思是：

```text
当 AI 不适合继续处理时，
把任务整理清楚，交给人。
```

你可以让系统生成一段人工交接摘要，包含：

- 用户想做什么
- 目前已知信息
- AI 为什么不能继续处理
- 需要人工确认什么
- 建议下一步怎么做

比如，当用户的问题涉及合同、报价、权限、客户敏感信息时，
系统不要直接回答，而是生成一份交接摘要。

你可以挑战自己

把现在的流程：

```text
用户输入
↓
input_guardrail
↓
agent_core.py
↓
最终回答
```

升级成：

```text
用户输入
↓
input_guardrail
↓
可以处理 → agent_core.py
信息不足 / 工具失败 → fallback
敏感 / 需要人工判断 → handoff
↓
最终回答
```


今天不要求你一定完成。

你只要先理解这个方向就很好：

```text
guardrail 让 AI 知道边界。
fallback 让 AI 在失败时不硬撑。
handoff 让 AI 在不适合继续时交给人。
```

# Part 4：用 Gradio 做成最终 Demo

现在，职业数字人的核心流程已经完成了。

它包含两层：

第一层是 `guardrail.py`。

它负责在用户输入进入主流程之前，先做一个最小输入护栏。

比如：

- 电话
- 邮箱

如果用户请求查看、输出或发送这类隐私信息，系统会先拦截。

第二层是 `agent_core.py`。

如果输入安全，就进入职业数字人的主流程：

```text
判断任务类型
↓
检查参数是否齐全
↓
调用工具
↓
读取资料和上下文
↓
生成回答
```

所以最终 Demo 不是只展示某一个模块。

它展示的是一个完整流程：

```text
用户输入
↓
guardrail.py：先做输入护栏
↓
安全 → agent_core.py：进入职业数字人主流程
不安全 → 直接拦截并提示
↓
页面显示结果
```

---

## Gradio 是什么？

到这里，大家已经知道怎么调用 `guardrail.py` 和 `agent_core.py` 了。

但现在还有一个问题：

这些能力都还在 Notebook 里。

如果要拿给别人看，比如同事、老板、客户，  
让他们直接看 Notebook 并不直观。

所以我们需要一个简单页面。

`Gradio` 就是一个可以快速把 Python 函数变成网页界面的工具。

你可以先把它理解成：

```text
Gradio = 给 Python 函数快速加一个网页页面
```

它特别适合做 AI Demo。

因为很多 AI 项目，本质上就是：

```text
用户输入一句话
↓
Python 函数处理
↓
返回 AI 的回答
```

而 Gradio 可以帮我们把这个过程变成一个简单网页：

```text
输入框
↓
提交按钮
↓
输出区域
```

这样，别人不需要打开代码，  
只要在网页里输入问题，就能体验你的职业数字人。

---

## 今天我们做的最小 Gradio 页面

今天先不做复杂前端。

我们只做一个最小版本：

```text
用户输入一个任务
↓
点击提交
↓
调用完整职业数字人流程
↓
页面显示回复
```

这里页面最终调用的是：

```python
guardrail.safe_handle_user_task(user_input)
```

注意：

虽然这里调用的是 `guardrail.safe_handle_user_task()`，  
但它不是只运行 guardrail。

它的内部逻辑是：

```text
先做 input guardrail
↓
如果安全，再调用 agent_core.handle_user_task()
```

所以这个函数代表的是：

```text
guardrail + agent_core 的完整职业数字人流程
```

这样，职业数字人就不只是 Notebook 里的代码，  
而是一个可以打开、可以输入、可以演示的 Demo。


In [1]:
import gradio as gr
import importlib

import src.agent_core as agent_core
import src.guardrail as guardrail

agent_core = importlib.reload(agent_core)
guardrail = importlib.reload(guardrail)

print("agent_core.py 和 guardrail.py 导入成功。")

c:\Users\240411014\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


agent_core.py 和 guardrail.py 导入成功。


In [2]:
def chat_with_career_agent(message, history):
    """
    Gradio ChatInterface 调用的函数。

    参数：
    - message：用户这一次输入的内容
    - history：前面的聊天历史，由 Gradio 自动传入

    这里我们先把 message 交给完整职业数字人流程：
    guardrail.py → agent_core.py
    """

    if not message or not message.strip():
        return "请输入一个任务。"

    try:
        result = guardrail.safe_handle_user_task(message)
        return result

    except Exception as e:
        return f"程序运行出错：{type(e).__name__}: {e}"

In [4]:
demo = gr.ChatInterface(
    fn=chat_with_career_agent,
    title="职业数字人 Demo",
    description="""
    您好，欢迎来到我的职业数字人demo聊天页面。

    您可以测试：
    - 你是谁？你做过什么？
    - 帮我写一段周报开头。
    - 请分析 data/sales_data.csv，告诉我三个重点。
    - 请帮我看这个网页主要讲什么：https://www.python.org/
    - 请告诉我客户电话。
    """,
    examples=[
        "你是谁？你做过什么？",
        "帮我写一段周报开头。",
        "请分析 data/sales_data.csv，告诉我三个重点。",
        "请帮我看这个网页主要讲什么：https://www.python.org/",
        "请告诉我客户电话。"
    ],
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


# Lab5 总结：职业数字人 Final 完成了

到这里，Lab5 就完成了。

我们把前四天的能力全部接了起来：

```text
Day 1：llm.py
接上大模型，让职业数字人有了“大脑”。

Day 2：task_router.py
让它能先判断任务类型。

Day 3：tool.py
让它能调用工具，读取表格、读取网页、推送消息。

Day 4：context.py
让它能读取身份资料、history、memory 和 resource。

Day 5：guardrail.py + Gradio
让它有了最小输入护栏，并做成可以展示的聊天页面。
```
现在，职业数字人已经具备一个最小完整系统的样子：

```text
用户输入
↓
guardrail.py
先判断是否涉及电话、邮箱等隐私信息
↓
agent_core.py
进入职业数字人主流程
↓
task_router.py
判断任务类型
↓
tool.py / context.py / llm.py
调用工具、读取资料、生成回答
↓
Gradio ChatInterface
在网页聊天页面里展示结果
```

这不是一个完整产品，
但它已经是一个可以运行、可以测试、可以继续扩展的 AI 工作助手原型。
---
## 你现在已经完成了一个重要转变

刚开始时，我们只是在调用一个大模型。

现在，你已经开始做系统了。

区别在于：

只调用模型：
问一句，答一句。

做职业数字人：
先判断任务，
再选择路径，
必要时调用工具，
需要时读取资料，
最后在边界内输出结果。

这就是从“会用 AI”到“会设计 AI 工作流”的第一步。

## 拓展练习

下面这些练习不要求今天全部完成。

你可以选 1 到 2 个继续做，
把这个职业数字人变得更像真实工作里的助手。

### 练习 1：让 Gradio 页面支持文件上传

现在页面只有聊天输入框。

你可以尝试给 Gradio 加一个文件上传组件，
让用户直接上传 CSV 或 Excel 文件。

目标流程：

```text
用户上传文件
↓
用户输入：请分析这份表格
↓
系统调用 full_data_analysis_tool()
↓
页面展示分析结果
```

### 练习 2：让 ChatInterface 真正使用 history

现在 Gradio 的 history 参数还没有被深入使用。

你可以尝试把 Gradio 传进来的聊天历史，
写入 data/context/history/，
或者临时拼进 prompt 里。

这样用户连续追问时，
职业数字人会更自然地接住上下文。

### 练习 6：把项目整理成可发布版本

你可以检查项目结构是否完整：

```python
ai-agent-course/
├── lab5.ipynb
├── src/
│   ├── __init__.py
│   ├── llm.py
│   ├── task_router.py
│   ├── tool.py
│   ├── context.py
│   ├── agent_core.py
│   └── guardrail.py
├── data/
│   ├── cv.txt
│   ├── work_log.txt
│   ├── sales_data.csv
│   └── context/
├── .env
├── .gitignore
└── requirements.txt
```

确认：
```python
.env 不上传
__pycache__/ 不上传
requirements.txt 已经补全
README.md 能说明项目怎么运行
```